# Brand Manager in a Box: fine-tune Gemma 4 12B

**Before you start:** Runtime > Change runtime type > **A100 GPU**, then Runtime > **Run all**.

Keep this tab open until the last cell finishes (about 2 to 3 hours). Progress is saved to your Google Drive folder `BrandManagerAI`, so a disconnect only loses the last few minutes.

In [ ]:
# True = 10-minute test run on the small 'smoke' set. False = the real training run.
SMOKE_TEST = False
from google.colab import drive
drive.mount('/content/drive')
DIR = '/content/drive/MyDrive/BrandManagerAI' + ('/smoke' if SMOKE_TEST else '')
import os; print(sorted(os.listdir(DIR)))

In [ ]:
%%capture
!pip install -U unsloth
!pip install 'transformers==5.10.2'

## 1. Load the starter brain
16-bit LoRA: we train a small add-on (about 1% of the model) instead of the whole thing.

In [ ]:
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = 'unsloth/gemma-4-12b-it',
    max_seq_length = 2048,
    load_in_4bit = False,
    full_finetuning = False,
    use_gradient_checkpointing = 'unsloth',
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = 'none', random_state = 3407,
)
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = 'gemma-4')

## 2. Load the flashcards

In [ ]:
from datasets import load_dataset
data = load_dataset('json', data_files = {s: f'{DIR}/{s}.jsonl' for s in ['train', 'val', 'test']})
def to_text(batch):
    return {'text': [tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False).removeprefix('<bos>') for m in batch['messages']]}
train = data['train'].map(to_text, batched = True)
val = data['val'].map(to_text, batched = True)
print(train, val)
print(train[0]['text'][:1500])

## 3. Answers from the untrained model (the 'before' picture)
Saved now, so the head-to-head compares against the same starter brain.

In [ ]:
import json, torch
test = data['test'].shuffle(seed = 7).select(range(3 if SMOKE_TEST else 40))
MAX_NEW = 150 if SMOKE_TEST else 500
def answer(messages, max_new_tokens = None):
    max_new_tokens = max_new_tokens or MAX_NEW
    prompt = [m for m in messages if m['role'] != 'assistant']
    inputs = tokenizer.apply_chat_template(prompt, add_generation_prompt = True, tokenize = True,
                                           return_dict = True, return_tensors = 'pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens = max_new_tokens, use_cache = True,
                         temperature = 1.0, top_p = 0.95, top_k = 64)
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens = True).strip()
    return text.removeprefix('thought').strip()  # empty thinking-channel marker left by the template
FastModel.for_inference(model)
with model.disable_adapter():
    before = [answer(r['messages']) for r in test]
json.dump(before, open(f'{DIR}/answers_before.json', 'w'))
print(before[0])

## 4. Train
One pass through every flashcard. Watch `loss` fall in the log. Checkpoints save to Drive every 200 steps.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
FastModel.for_training(model)
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = train, eval_dataset = val,
    args = SFTConfig(
        dataset_text_field = 'text',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.03,
        num_train_epochs = 1,
        max_steps = 10 if SMOKE_TEST else -1,
        learning_rate = 1e-4,
        lr_scheduler_type = 'cosine',
        logging_steps = 10,
        eval_strategy = 'steps', eval_steps = 5 if SMOKE_TEST else 200,
        save_strategy = 'steps', save_steps = 5 if SMOKE_TEST else 200, save_total_limit = 2,
        output_dir = f'{DIR}/checkpoints',
        optim = 'adamw_8bit', weight_decay = 0.01, seed = 3407,
        report_to = 'none',
    ),
)
trainer = train_on_responses_only(trainer, instruction_part = '<|turn>user\n', response_part = '<|turn>model\n')
import glob
resume = bool(glob.glob(f'{DIR}/checkpoints/checkpoint-*'))
stats = trainer.train(resume_from_checkpoint = resume)
print(stats)

## 5. Save the trained add-on to Drive

In [ ]:
model.save_pretrained(f'{DIR}/brand-manager-lora')
tokenizer.save_pretrained(f'{DIR}/brand-manager-lora')
print('saved')

## 6. Answers from the trained model (the 'after' picture)

In [ ]:
FastModel.for_inference(model)
after = [answer(r['messages']) for r in test]
rows = [{'request': [m for m in r['messages'] if m['role'] == 'user'][0]['content'],
         'reference': [m for m in r['messages'] if m['role'] == 'assistant'][0]['content'],
         'before': b, 'after': a} for r, b, a in zip(test, before, after)]
json.dump(rows, open(f'{DIR}/head_to_head.json', 'w'), indent = 2)
for r in rows[:2]:
    print('REQUEST:', r['request'][:400], '\n\nBEFORE:', r['before'][:600], '\n\nAFTER:', r['after'][:600], '\n' + '=' * 80)

## 7. Try it yourself
Type your own brand question below and run the cell.

In [ ]:
question = 'We are launching a premium dog food at twice the price of the category leader. How do we justify the price?'
print(answer([{'role': 'system', 'content': train[0]['messages'][0]['content']},
              {'role': 'user', 'content': question}]))